In [ ]:
#this code is reading the given final clean file and then classifying the patients based on tiers and feeding this data to the ml model and patterns are leanrt 
#then the data is stored in a new piclke file

In [2]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

print("1. Loading compressed master dataset...")
df = pd.read_csv("final_clean_master.csv")

print("2. Engineering Triage Logic (Strict Risk Tiers)...")
def assign_tier(row):
    # Tier 1 (Critical): Multiple ICU visits OR massive lab abnormalities combined with terrible adherence
    if row['icu_admissions'] > 1 or (row['abnormal_labs'] > 15 and row['avg_adherence'] < 60):
        return 0 
    # Tier 2 (Moderate): Some ICU history, moderate lab issues, or dropping adherence
    elif row['icu_admissions'] == 1 or row['abnormal_labs'] > 8 or row['avg_adherence'] < 75:
        return 1
    # Tier 3 (Safe): Normal limits
    else:
        return 2

df['risk_tier'] = df.apply(assign_tier, axis=1)

print("3. Selecting Features for the AI...")
features = [
    'age', 'bmi', 'systolic_bp', 'diastolic_bp', 'heart_rate', 
    'avg_adherence', 'total_meds', 'abnormal_labs', 'total_tests'
]

X = df[features].fillna(0)
y = df['risk_tier']

print("4. Splitting Data for Testing...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("5. Training the XGBoost Brain...")
model = xgb.XGBClassifier(objective='multi:softprob', num_class=3, eval_metric='mlogloss')
model.fit(X_train, y_train)

print("6. Evaluating Accuracy...")
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Tier 1 (Call)', 'Tier 2 (SMS)', 'Tier 3 (None)']))

print("7. Saving the Model...")
joblib.dump(model, 'xgboost_triage_model.pkl')
print("CRITICAL SUCCESS: Model physically saved as xgboost_triage_model.pkl")

1. Loading compressed master dataset...
2. Engineering Triage Logic (Strict Risk Tiers)...
3. Selecting Features for the AI...
4. Splitting Data for Testing...
5. Training the XGBoost Brain...
6. Evaluating Accuracy...
               precision    recall  f1-score   support

Tier 1 (Call)       1.00      1.00      1.00        36
 Tier 2 (SMS)       1.00      1.00      1.00     10887
Tier 3 (None)       1.00      1.00      1.00      9077

     accuracy                           1.00     20000
    macro avg       1.00      1.00      1.00     20000
 weighted avg       1.00      1.00      1.00     20000

7. Saving the Model...
CRITICAL SUCCESS: Model physically saved as xgboost_triage_model.pkl
